# Taller — Predicción de accidentalidad
**Aprendizaje Automático** · Clasificación binaria con clases desbalanceadas

**Integrantes:** _escriba aquí los nombres_

---

### Cómo usar este notebook
- Este notebook sigue la estructura del enunciado del taller (PDF). Léalo antes de empezar.
- Las celdas que ya tienen código completo **no necesitan cambios**: ejecútelas tal cual.
- Donde vea `...` o un comentario `# TODO`, **usted debe completar el código**.
- Las líneas que empiezan con `# Pista:` le dan una idea de cómo resolverlo. No es obligatorio seguirlas.
- Las celdas **✍️ Respuesta** son para sus respuestas escritas (doble clic para editarlas).
- Antes de entregar: *Kernel → Restart & Run All*. El notebook debe ejecutarse completo y sin errores.
- El archivo `accidentes_taller.sqlite` debe estar en la misma carpeta que este notebook.

## 0. Librerías y configuración

In [ ]:
# Librerías que necesitará en el taller (no hace falta instalar nada adicional si tiene Anaconda o Colab)
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn   # se requiere scikit-learn 1.2 o superior

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score,
                             f1_score, accuracy_score, precision_recall_curve,
                             confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42  # úselo en todo lo que tenga aleatoriedad
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (9, 4)

print('pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

---
## Parte 1 · Carga y calidad de los datos (15 pts)

### 1.0 Carga de las tablas
Esta celda ya está completa: se conecta a la base SQLite y carga las dos tablas con SQL.

In [ ]:
con = sqlite3.connect('accidentes_taller.sqlite')
accidentes = pd.read_sql('SELECT * FROM accidentes', con, parse_dates=['FECHA_HORA'])
clima      = pd.read_sql('SELECT * FROM clima', con, parse_dates=['TW'])
con.close()

print('accidentes:', accidentes.shape)
print('clima:     ', clima.shape)

In [ ]:
accidentes.head()

In [ ]:
clima.head()

### 1.1 Nombres de barrio inconsistentes
La siguiente celda muestra los nombres de barrio que aparecen en `accidentes` pero **no** en `clima`.
Ejecútela y observe qué tienen de raro esos nombres.

In [ ]:
barrios_clima = set(clima['BARRIO'].unique())
no_coinciden = sorted(set(accidentes['BARRIO'].unique()) - barrios_clima)
print(f'{len(no_coinciden)} nombres de barrio en accidentes no aparecen en clima:')
print(no_coinciden)

In [ ]:
# TODO: corrija la columna BARRIO de `accidentes` para que use exactamente los mismos nombres que `clima`.
# Pista: construya un diccionario {nombre en minúsculas: nombre oficial} a partir de los barrios de `clima`,
#        y luego aplique .str.strip().str.lower() y .map(diccionario) a la columna de accidentes.
# Pista: reporte cuántas filas cambiaron.

accidentes['BARRIO'] = ...

# Verificación: después de corregir, esto debería imprimir 0 y no debería haber nulos en BARRIO
print(len(set(accidentes['BARRIO'].unique()) - barrios_clima), 'nombres sin corregir')
print(accidentes['BARRIO'].isna().sum(), 'nulos en BARRIO')

### 1.2 Registros duplicados

In [ ]:
# TODO: encuentre y elimine los accidentes duplicados.
# Pista: pruebe primero accidentes.duplicated().sum(). ¿Da 0? Mire la columna ID:
#        es distinta en cada fila, así que no sirve para detectar duplicados.
#        Use el parámetro subset= con las columnas que describen el accidente.

columnas_accidente = [...]   # complete con los nombres de las columnas
n_duplicados = ...
print('Duplicados encontrados:', n_duplicados)

accidentes = ...
print('Filas después de eliminar duplicados:', len(accidentes))

### 1.3 Valores faltantes e imposibles en el clima

In [ ]:
print(clima.isna().sum())
clima.describe()

In [ ]:
# TODO: trate los valores imposibles y los faltantes de las variables de clima.
# Pista: en describe(), mire el valor mínimo de temperatura. ¿Es físicamente posible en esta ciudad?
#        Convierta esos valores imposibles en NaN (np.nan) y luego impute todos los faltantes.
# Pista: una opción simple es imputar con la mediana. Para no usar información del futuro,
#        calcule la mediana solo con los datos de entrenamiento (antes de '2025-04-01').

...

# Verificación: no deberían quedar nulos ni temperaturas imposibles
print(clima[['temperatura', 'humedad']].isna().sum())
print('Temperatura mínima:', clima['temperatura'].min())

**✍️ Respuesta — Parte 1: ¿qué problemas encontró, cuántos registros afectaba cada uno y cómo los resolvió?**

_Escriba aquí su respuesta (2 a 5 líneas)._

---
## Parte 2 · Construcción de la variable objetivo (15 pts)

Recuerde: `clima` tiene **todas** las parejas (barrio, hora); `accidentes` solo tiene las parejas donde sí hubo accidente.

In [ ]:
# TODO: cree la columna TW en `accidentes` truncando FECHA_HORA a la hora.
# Pista: .dt.floor('h')

accidentes['TW'] = ...
accidentes[['FECHA_HORA', 'TW']].head()

In [ ]:
# TODO: construya la tabla de positivos: una fila por cada pareja (BARRIO, TW) con al menos un accidente.
# Pista: seleccione las columnas BARRIO y TW, elimine duplicados y agregue una columna target = 1.

positivos = ...
print('Parejas (barrio, hora) con al menos un accidente:', len(positivos))

In [ ]:
# TODO: una `clima` con `positivos` para obtener una fila por (barrio, hora) con la columna target (0/1).
# Pista: piense bien qué tabla va a la izquierda y qué tipo de join usar (how=...).
#        Después del join, las filas sin accidente quedan con target NaN: reemplácelas por 0.

df = ...

# Verificación: debe haber exactamente una fila por cada fila de clima
assert len(df) == len(clima), 'El número de filas no coincide con clima: revise el join'
print(f"Filas: {len(df)} | Porcentaje de positivos: {df['target'].mean():.2%}")

**✍️ Respuesta — ¿Qué join usó y por qué? ¿Qué pasaría con un inner join? ¿Qué implica el porcentaje de positivos?**

_Escriba aquí su respuesta (2 a 5 líneas)._

---
## Parte 3 · Análisis exploratorio (15 pts)

Haga exactamente tres gráficos. El primero está resuelto como ejemplo; úselo como modelo para los otros dos.

In [ ]:
# Gráfico 1 (ejemplo resuelto): proporción de horas con accidente según la hora del día
tasa_por_hora = df.groupby(df['TW'].dt.hour)['target'].mean()

tasa_por_hora.plot(kind='bar', color='steelblue')
plt.title('Proporción de horas con accidente según la hora del día')
plt.xlabel('Hora del día')
plt.ylabel('Proporción con accidente')
plt.show()

**✍️ Respuesta — Conclusión del gráfico 1**

_Escriba aquí su respuesta (2 a 5 líneas)._

In [ ]:
# TODO: Gráfico 2 — proporción de horas con accidente por barrio, ordenada de mayor a menor.
# Pista: agrupe por BARRIO, use .sort_values(ascending=False) y un gráfico de barras.
#        Si los nombres no caben, pruebe kind='barh' o rot=90.

...

**✍️ Respuesta — Conclusión del gráfico 2**

_Escriba aquí su respuesta (2 a 5 líneas)._

In [ ]:
# TODO: Gráfico 3 — proporción de horas con accidente con lluvia vs. sin lluvia.
# Pista: agrupe por la condición (df['precipitacion'] > 0) y renombre las categorías
#        para que el gráfico diga 'Sin lluvia' y 'Con lluvia' en lugar de False/True.

...

**✍️ Respuesta — Conclusión del gráfico 3**

_Escriba aquí su respuesta (2 a 5 líneas)._

---
## Parte 4 · Variables (features) (10 pts)

In [ ]:
# TODO: cree las variables temporales y de lluvia.
# Pista: df['TW'].dt.hour, df['TW'].dt.dayofweek (0 = lunes, 6 = domingo)

df['hora'] = ...
df['dia_semana'] = ...
df['fin_de_semana'] = ...   # 1 si es sábado o domingo, 0 si no
df['llueve'] = ...          # 1 si precipitacion > 0, 0 si no

df[['TW', 'hora', 'dia_semana', 'fin_de_semana', 'llueve']].head()

In [ ]:
# OPCIONAL (bono): codificación cíclica de la hora y del día de la semana.
# Pista: hora_sin = sin(2·π·hora/24), hora_cos = cos(2·π·hora/24). Lo mismo con el día de la semana usando 7.

In [ ]:
# TODO: codifique BARRIO con one-hot encoding y agréguelo a df.
# Pista: pd.get_dummies(df['BARRIO'], prefix='barrio', dtype=int) y luego pd.concat([...], axis=1)

dummies_barrio = ...
df = ...

In [ ]:
# TODO: defina la lista de columnas que usará como variables del modelo.
# Pista: incluya las variables que creó, las dummies de barrio (list(dummies_barrio.columns))
#        y las variables de clima que su análisis exploratorio justifique.
#        NO incluya TW, BARRIO, COMUNA ni target.

features = [...]

print(len(features), 'variables')

**✍️ Respuesta — ¿Qué variables de clima incluyó y por qué?**

_Escriba aquí su respuesta (2 a 5 líneas)._

---
## Parte 5 · Modelado con clases desbalanceadas (30 pts)

### 5.0 Partición temporal
Entrenamiento: enero a marzo. Prueba (test): abril.

In [ ]:
# TODO: ordene df por TW y divídalo en entrenamiento (TW < '2025-04-01') y prueba (TW >= '2025-04-01').
# Pista: es importante ORDENAR por TW antes de dividir: TimeSeriesSplit (sección 5c) asume que las filas
#        están en orden cronológico.

df = ...
corte = '2025-04-01'
train = ...
test = ...

X_train, y_train = train[features], train['target']
X_test, y_test = test[features], test['target']
print(f'Entrenamiento: {len(train)} filas, {y_train.mean():.2%} positivos')
print(f'Prueba:        {len(test)} filas, {y_test.mean():.2%} positivos')

### Función auxiliar para evaluar modelos
Esta función ya está completa. Recibe las probabilidades predichas y devuelve un diccionario con todas las métricas
que pide el taller. Guarde cada resultado en la lista `resultados` para armar la tabla comparativa al final.

In [ ]:
resultados = []

def evaluar_modelo(nombre, y_true, proba, umbral=0.5):
    """Calcula las métricas del taller a partir de las probabilidades de la clase positiva."""
    y_pred = (proba >= umbral).astype(int)
    return {
        'modelo': nombre,
        'PR-AUC': average_precision_score(y_true, proba),
        'ROC-AUC': roc_auc_score(y_true, proba),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
    }

# Ejemplo de uso:
# proba = modelo.predict_proba(X_test)[:, 1]
# resultados.append(evaluar_modelo('Nombre del modelo', y_test, proba))

### 5a. Baseline: siempre predecir 0

In [ ]:
# TODO: calcule la accuracy en test de un modelo que siempre predice 0 (nunca hay accidente).
# Pista: basta con comparar y_test con un arreglo de ceros, o usar np.zeros(len(y_test)).

accuracy_baseline = ...
print(f'Accuracy del baseline: {accuracy_baseline:.2%}')

**✍️ Respuesta — ¿Por qué este número es engañoso?**

_Escriba aquí su respuesta (2 a 5 líneas)._

### 5b. Regresión logística sin y con balanceo

In [ ]:
# TODO: entrene una regresión logística SIN balanceo, calcule las probabilidades en test y guarde sus métricas.
# Pista: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
#        proba = modelo.predict_proba(X_test)[:, 1]

...

In [ ]:
# TODO: repita con class_weight='balanced' y guarde sus métricas.

...

### 5c. Modelo de ensamble con ajuste de hiperparámetros

In [ ]:
# TODO: elija un modelo de ensamble, defina una grilla pequeña y ajústela con GridSearchCV + TimeSeriesSplit.
# Pista: HistGradientBoostingClassifier es rápido y acepta class_weight='balanced'.
#        Hiperparámetros sugeridos: 'learning_rate', 'max_depth' o 'max_leaf_nodes'.
#        Si usa RandomForestClassifier: 'max_depth', 'min_samples_leaf' (y n_estimators ≈ 200).
# Límite: máximo 2 hiperparámetros con 3 valores cada uno.

modelo_base = ...

grilla = {
    # 'nombre_del_hiperparametro': [valor1, valor2, valor3],
}

busqueda = GridSearchCV(
    modelo_base,
    param_grid=grilla,
    cv=TimeSeriesSplit(n_splits=3),
    scoring='average_precision',
    n_jobs=-1,
)
busqueda.fit(X_train, y_train)

print('Mejores hiperparámetros:', busqueda.best_params_)
print(f'PR-AUC promedio en validación: {busqueda.best_score_:.3f}')

In [ ]:
# TODO: evalúe en test el mejor modelo encontrado y guarde sus métricas.
# Pista: busqueda.best_estimator_ ya está reentrenado con todo el conjunto de entrenamiento.

...

### 5d. Tabla comparativa

In [ ]:
tabla = pd.DataFrame(resultados).set_index('modelo').round(3)
print(f'Referencia: un modelo al azar tendría PR-AUC ≈ {y_test.mean():.3f}')
tabla

**✍️ Respuesta — ¿Qué cambió al usar class_weight='balanced'? ¿Por qué la PR-AUC es más informativa que la ROC-AUC aquí?**

_Escriba aquí su respuesta (2 a 5 líneas)._

### (Opcional · bono) Submuestreo o tercer modelo
Pista: para el submuestreo, tome **solo del conjunto de entrenamiento** todos los positivos y una muestra aleatoria
de negativos (por ejemplo, 3 negativos por cada positivo) con `.sample(..., random_state=RANDOM_STATE)`.
Nunca submuestree el conjunto de prueba.

In [ ]:
# OPCIONAL

---
## Parte 6 · Umbral de decisión y caso de uso (15 pts)

**✍️ Respuesta — ¿Cuál es su mejor modelo y por qué?**

_Escriba aquí su respuesta (2 a 5 líneas)._

In [ ]:
# TODO: guarde en esta variable las probabilidades en test de su mejor modelo.
proba_final = ...

In [ ]:
# TODO: encuentre el umbral que logra un recall de al menos 0,60 con la mayor precisión posible.
# Pista: precision_recall_curve devuelve precision y recall con UN elemento más que umbrales.
#        Descarte el último elemento de precision y recall para que las longitudes coincidan.
#        Luego filtre los umbrales cuyo recall >= 0.60 y, entre ellos, quédese con el de mayor precisión.

precision, recall, umbrales = precision_recall_curve(y_test, proba_final)
precision, recall = precision[:-1], recall[:-1]

umbral_final = ...

y_pred_final = (proba_final >= umbral_final).astype(int)
print(f'Umbral elegido: {umbral_final:.3f}')
print(f'Recall: {recall_score(y_test, y_pred_final):.3f} | Precisión: {precision_score(y_test, y_pred_final):.3f}')

In [ ]:
# TODO: muestre la matriz de confusión con el umbral elegido.
# Pista: ConfusionMatrixDisplay.from_predictions(y_test, y_pred_final, display_labels=['Sin accidente', 'Accidente'])

...

**✍️ Respuesta — Interprete en palabras los falsos positivos y los falsos negativos**

_Escriba aquí su respuesta (2 a 5 líneas)._

### Caso de uso: Top-10 de riesgo para un día de abril

In [ ]:
# TODO: para un día de abril, liste las 10 parejas (barrio, hora) con mayor probabilidad predicha
#       y cuente cuántas tuvieron realmente un accidente.
# Pista: agregue proba_final como columna a una copia de test, filtre por la fecha con
#        test['TW'].dt.date == pd.Timestamp('2025-04-10').date() y use .nlargest(10, 'proba').

dia = '2025-04-10'   # puede cambiar la fecha
top10 = ...

print('Aciertos en el Top-10:', ...)
top10

**✍️ Respuesta — Limitaciones del modelo y una variable externa que lo mejoraría**

_Escriba aquí su respuesta (2 a 5 líneas)._

---
## ✅ Lista de chequeo antes de entregar
- [ ] *Kernel → Restart & Run All* se ejecuta completo y sin errores.
- [ ] No quedan `...` ni `# TODO` sin resolver.
- [ ] Todas las celdas ✍️ Respuesta están contestadas.
- [ ] Los tres gráficos tienen título y ejes rotulados.
- [ ] La tabla comparativa de la Parte 5 tiene al menos tres modelos (logística sin y con balanceo, y el ensamble).
- [ ] Los nombres de los integrantes están al inicio del notebook.